# cellmap-flow on Colab

Runs `cellmap_flow_server` on this Colab session and exposes it via a
public Cloudflare Tunnel. Prints a Neuroglancer link that anyone can
open to see the raw EM alongside the inference output.

Same architecture as running `cellmap_flow huggingface --repo ...
-d ...` on a Janelia workstation: one inference Flask app, NG (via
neuroglancer-demo.appspot.com) fetches chunks from it. No dashboard
process needed for the demo itself — if you want the Input/Postprocess
pipeline editor, run `cellmap_flow_app` separately.

Free Colab gives you a T4 GPU — ~10× faster than HF Space CPU. URL is
only alive while this session is.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run all cells.
3. Open the printed Neuroglancer link.
4. Pan / zoom. Chunks compute on the Colab T4.

## 1. Install cellmap-flow + cloudflared

Takes ~3–5 min the first time. cloudflared is a tiny static binary;
no signup needed.

In [ ]:
# cellmap-flow with the bioimageio extra (so `bioimage` works too).
%pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs

# Pin bioimageio.spec to a version that still parses v0.4-style RDFs
# (hiding-blowfish + most 2D BMZ models still ship those). The default
# resolver on Python 3.12 picks a newer spec that drops v0.4 support.
%pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"

# cloudflared static binary for the public tunnel.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared


## 2. Configure model + dataset

Pick one of two model types:
- `huggingface` — a cellmap HF model (e.g. `cellmap/fly_organelles_run07_432000`).
- `bioimage` — a BioImage Model Zoo model (e.g. `hiding-blowfish`).

Set `MODEL_TYPE` to switch.

> **T4 sizing**: 178³-input HF models (`fly_organelles_run07_*`) fit T4
> cleanly (~5–6 GB peak). 288³ models (`jrc_mus-livers_*`) are
> borderline. 378³ models (`mito-aff-unet-*`) won't fit. BMZ 2D models
> like `hiding-blowfish` are tiny (~200 MB peak).

In [ ]:
MODEL_TYPE = "huggingface"   # or "bioimage"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"  # 178^3 inference; fits T4
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)
# Other T4-friendly options:
#   HF_REPO = "cellmap/fly_organelles_run07_700000"
#   HF_REPO = "cellmap/fly_organelles_run08_438000"
#   HF_REPO = "cellmap/jrc_mus-livers_16nm_to_8nm_mito"  # BORDERLINE, OOM-prone

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"     # nm per voxel
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

PORT = 8765

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")


## 3. Start the cellmap-flow inference server

Runs in the background so cloudflared can come up alongside.

In [ ]:
import os, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO,
        "--name", HF_NAME,
        "-d", HF_DATASET,
        "--port", str(PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL,
        "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET,
        "--port", str(PORT),
    ]

print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{PORT}" in line:
        print("\n[server] ready.")
        break


## 4. Public cloudflared tunnel

In [ ]:
import subprocess, re, time

tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break

print("\n" + "=" * 70)
print(f"BACKEND URL: {PUBLIC_URL}")
print("=" * 70)


## 5. Build a Neuroglancer link

Same encoded-state link `cellmap_flow` prints when you run it locally:
raw EM in one layer, inference in another, pointed at this Colab T4 via
the public cloudflared URL.

For BMZ models, the raw URL is stripped to the parent multiscale group
(so NG handles its own mip selection + gets z/y/x dim labels rather
than d0/d1/d2).

In [ ]:
import json, urllib.parse, re

# Strip trailing /sN from DATASET when building the raw NG layer so NG
# opens the multiscale group (proper axes + mip selection). The backend
# still operates on whatever level DATASET points at.
raw_url = re.sub(r"/s\d+/?$", "", DATASET)
voxel_nm = 8

state = {
    "dimensions": {
        "z": [voxel_nm * 1e-9, "m"],
        "y": [voxel_nm * 1e-9, "m"],
        "x": [voxel_nm * 1e-9, "m"],
    },
    "layers": [
        {"type": "image", "source": f"zarr://{raw_url}", "name": "raw"},
        {
            "type": "image",
            "source": f"zarr://{PUBLIC_URL}/{MODEL_NAME}/",
            "name": MODEL_NAME,
        },
    ],
    "selectedLayer": {"visible": True, "layer": MODEL_NAME},
    "layout": "4panel",
}
encoded = urllib.parse.quote(json.dumps(state, separators=(",", ":")))
ng_url = f"https://neuroglancer-demo.appspot.com/#!{encoded}"

print("\n" + "*" * 70)
print("DEMO URL (open in any browser):")
print(ng_url)
print("*" * 70)
print()
print("Direct cellmap-flow inference server (for use with other tools):")
print(f"  {PUBLIC_URL}")


## 6. Keep-alive

Run this cell last so Colab doesn't idle-disconnect. Stop it with ▢
when you're done — that also closes the tunnel.

In [ ]:
import time, select

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        drain(tunnel, "tunnel")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}.")
            break
        if tunnel.poll() is not None:
            drain(tunnel, "tunnel")
            print(f"\n[tunnel] exited rc={tunnel.returncode}.")
            break
        time.sleep(2)
finally:
    for p in (server, tunnel):
        try: p.terminate()
        except Exception: pass
